In [10]:
# ==========================================
# CELL 1: CLEAN ENVIRONMENT SETUP
# ==========================================
print("🔧 Setting up clean environment...")

# 1. Uninstall conflicting packages
!pip uninstall -y numpy scipy pandas scikit-learn jax jaxlib opencv-python opencv-contrib-python opencv-python-headless --yes -q 2>/dev/null

# 2. Install compatible numpy FIRST
!pip install "numpy>=2.0,<2.1" -q

# 3. Install PyTorch
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 -q

# 4. Install Transformers stack
!pip install transformers accelerate bitsandbytes -q

# 5. Install LangChain with correct versions
!pip install "langchain>=0.1.0" "langchain-community>=0.0.20" "langchain-core>=0.1.0" -q
!pip install langchain-huggingface chromadb sentence-transformers -q

# 6. Mount Google Drive
from google.colab import drive
import os

if not os.path.exists('/content/drive'):
    print("📂 Mounting Drive...")
    drive.mount('/content/drive')
else:
    print("✅ Drive already mounted.")

print("\n✅ Setup complete!")
print("⚠️  IMPORTANT: Go to Runtime → Restart session, then run Cell 2")


🔧 Setting up clean environment...
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
orbax-checkpoint 0.11.28 requires jax>=0.6.0, which is not installed.
matplotlib-venn 1.1.2 requires scipy, which is not installed.
lightgbm 4.6.0 requires scipy, which is not installed.
osqp 1.0.5 requires scipy>=0.13.2, which is not installed.
dask-cuda 25.10.0 requires pandas>=1.3, which is not installed.
bigframes 2.29.1 requires pandas>=1.5.3, which is not installed.
hyperopt 0.2.7 requires scipy, which is not installed.
db-dtypes 1.4.4 requires pandas>=1.5.3, which is not installed.
pysal 25.7 requires pandas>=1.4, which is not installed.
pysal 25.7 requires scikit-learn>=1.1, which is not installed.
pysal 25.7 requires scipy>=1.8, which is not installed.
imbalanced-learn 0.14.0 requires scikit-learn<2,>=1.4.2, which is not installed.
imbalanced-learn 0.14.0 requires scipy<2

In [2]:
pip install sentence-transformers

In [12]:
!pip install pandas scikit-learn scipy -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires jax>=0.1.72, which is not installed.
dopamine-rl 4.1.2 requires jaxlib>=0.1.51, which is not installed.
dopamine-rl 4.1.2 requires opencv-python>=3.4.8.29, which is not installed.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [5]:
# ==========================================
# CELL 2: LOAD RAG APPLICATION
# ==========================================
import torch
import os
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig

# --- CONFIGURATION ---
BASE_PATH = "/content/drive/MyDrive/GenAiProject_Dataset"
DB_PATH = os.path.join(BASE_PATH, "vector_db_advanced_500")

print("="*70)
print("🚀 LOADING RAG SYSTEM")
print("="*70)

# 1. Load Vector Database
print("\n[1/3] 🧠 Connecting to Vector Database...")

if not os.path.exists(DB_PATH):
    print(f"❌ ERROR: Database not found at {DB_PATH}")
    print("   Please check the path or run the vector creation notebook first.")
    raise FileNotFoundError(f"Vector database not found: {DB_PATH}")

emb = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5",
    model_kwargs={'device': 'cuda'},
    encode_kwargs={'normalize_embeddings': True}
)

vectorstore = Chroma(
    persist_directory=DB_PATH,
    embedding_function=emb,
    collection_name="documents"
)

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

print("      ✅ Vector database loaded successfully")

# 2. Load Language Model
print("\n[2/3] 🔥 Loading LLM (Zephyr-7B-Beta)...")
print("      ⏳ This takes 2-3 minutes on first run...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

model_id = "HuggingFaceH4/zephyr-7b-beta"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

text_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.7,
    repetition_penalty=1.1,
    return_full_text=False,
    do_sample=True
)

llm = HuggingFacePipeline(pipeline=text_pipeline)

print("      ✅ Language model loaded successfully")

# 3. Build RAG Chain (Modern LangChain API)
print("\n[3/3] ⚙️  Building RAG pipeline...")

prompt_template = """### [INST]
You are an expert AI Study Assistant.
Use the provided context to answer the question accurately and concisely.
If the context doesn't contain the answer, say so clearly.

CONTEXT:
{context}

QUESTION:
{question}

[/INST]
"""

prompt = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

# Format documents for context
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Build the RAG chain using LCEL (LangChain Expression Language)
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("      ✅ RAG pipeline ready")

print("\n" + "="*70)
print("✅ SYSTEM READY - Run Cell 3 to ask questions!")
print("="*70)

🚀 LOADING RAG SYSTEM

[1/3] 🧠 Connecting to Vector Database...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

model-00007-of-00008.safetensors:   0%|          | 0.00/1.98G [00:00<?, ?B/s]

model-00002-of-00008.safetensors:   0%|          | 0.00/1.95G [00:00<?, ?B/s]

model-00003-of-00008.safetensors:   0%|          | 0.00/1.98G [00:00<?, ?B/s]

model-00008-of-00008.safetensors:   0%|          | 0.00/816M [00:00<?, ?B/s]

KeyboardInterrupt: 

In [6]:
# ==========================================
# CELL: ROBUST MODEL LOADER (Run this to resume)
# ==========================================
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig
from langchain_huggingface import HuggingFacePipeline

# 1. Configuration (4-bit)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

model_id = "HuggingFaceH4/zephyr-7b-beta"
print(f"🚀 Downloading {model_id}...")
print("   (resume_download=True is ACTIVE. It will pick up where it stuck.)")

# 2. Load Model with Resume Logic
try:
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto",
        resume_download=True,  # <--- THIS FIXES THE HANG
        trust_remote_code=True
    )
    print("✅ Model Loaded Successfully!")
except Exception as e:
    print(f"❌ Error: {e}")

# 3. Create Pipeline
text_gen_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.7,
    repetition_penalty=1.1,
    return_full_text=False,
    do_sample=True
)
llm = HuggingFacePipeline(pipeline=text_gen_pipeline)

print("✅ Pipeline Ready.")

🚀 Downloading HuggingFaceH4/zephyr-7b-beta...
   (resume_download=True is ACTIVE. It will pick up where it stuck.)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

model-00004-of-00008.safetensors:   0%|          | 0.00/1.95G [00:00<?, ?B/s]

model-00006-of-00008.safetensors:   0%|          | 0.00/1.95G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Device set to use cuda:0


✅ Model Loaded Successfully!
✅ Pipeline Ready.


In [20]:
# ==========================================
# BUILD RAG CHAIN (Run this now!)
# ==========================================
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

print("⚙️  Building RAG pipeline...")

prompt_template = """### [INST]
You are an expert AI Study Assistant.
Use the provided context to answer the question accurately and concisely.
If the context doesn't contain the answer, say so clearly.

CONTEXT:
{context}

QUESTION:
{question}

[/INST]
"""

prompt = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

# Format documents for context
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Build the RAG chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("✅ RAG chain created successfully!")
print(f"🔍 Verification: rag_chain = {type(rag_chain)}")

⚙️  Building RAG pipeline...
✅ RAG chain created successfully!
🔍 Verification: rag_chain = <class 'langchain_core.runnables.base.RunnableSequence'>


In [19]:
# ==========================================
# CELL 3: ASK QUESTIONS (FIXED)
# ==========================================

def ask(question):
    """
    Ask a question to the RAG system

    Args:
        question (str): Your question

    Returns:
        None (prints formatted answer with sources)
    """
    print("\n" + "="*70)
    print(f"❓ QUESTION: {question}")
    print("="*70)

    # Get relevant documents (using invoke for newer LangChain versions)
    docs = retriever.invoke(question)

    # Generate answer using rag_chain
    print("\n🤖 ANSWER:")
    print("-"*70)
    answer = rag_chain.invoke(question)
    print(answer)
    print("-"*70)

    # Show sources
    print("\n📚 SOURCES:")
    for i, doc in enumerate(docs, 1):
        source = os.path.basename(doc.metadata.get('source', 'Unknown'))
        preview = doc.page_content[:100].replace('\n', ' ')
        print(f"  [{i}] {source}")
        print(f"      Preview: {preview}...")
    print("="*70 + "\n")

# ==========================================
# EXAMPLE USAGE
# ==========================================

# Test Question 1
ask("What is the architecture of a Transformer?")

# Test Question 2
ask("Explain the attention mechanism in neural networks")


❓ QUESTION: What is the architecture of a Transformer?

🤖 ANSWER:
----------------------------------------------------------------------
The Transformer is an attention-layer-based, sequence-to-sequence ("Seq2Seq") encoder-decoder architecture that relies entirely on self-attention to compute representations of its input and output without using recurrent neural networks (RNNs) or convolutions. This architecture has emerged as a groundbreaking approach to various language-related tasks, replacing RNN-based approaches in many applications due to its ability to handle long sequences and parallelize computation across multiple sequences. The Transformer was introduced in the paper "Attention Is All You Need" by Vaswani et al. In 2017. It has shown state-of-the-art performance in numerous natural language processing tasks, including machine translation, question answering, and summarization.
----------------------------------------------------------------------

📚 SOURCES:
  [1] Lecture #

In [24]:
# ==========================================
# CELL 4: BASELINE VS ADVANCED COMPARISON
# ==========================================
import os
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

print("="*70)
print("🔄 SWITCHING TO BASELINE MODEL FOR COMPARISON")
print("="*70)

# --- BASELINE CONFIGURATION ---
BASELINE_DB_PATH = os.path.join(BASE_PATH, "vector_db_baseline_500")

print("\n[1/2] 📦 Loading Baseline (MiniLM) Retriever...")

# Check if baseline database exists
if not os.path.exists(BASELINE_DB_PATH):
    print(f"⚠️  WARNING: Baseline database not found at {BASELINE_DB_PATH}")
    print("   Creating baseline with MiniLM embeddings...")
    # You'll need to create this database first
else:
    print(f"   ✅ Found baseline database")

# Load Baseline Embeddings (MiniLM - smaller, less accurate)
baseline_emb = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={'device': 'cuda'},
    encode_kwargs={'normalize_embeddings': True}
)

baseline_vectorstore = Chroma(
    persist_directory=BASELINE_DB_PATH,
    embedding_function=baseline_emb,
    collection_name="documents"
)

baseline_retriever = baseline_vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

print("   ✅ Baseline retriever loaded")

print("\n[2/2] 🔗 Building Baseline Chain...")

# Use the same prompt template
baseline_prompt = PromptTemplate(
    template=prompt_template,  # Reusing from Cell 2
    input_variables=["context", "question"]
)

# Build Baseline Chain (same structure as advanced)
baseline_chain = (
    {"context": baseline_retriever | format_docs, "question": RunnablePassthrough()}
    | baseline_prompt
    | llm  # Using the same LLM (Zephyr) for fair comparison
    | StrOutputParser()
)

print("   ✅ Baseline chain ready")

# ==========================================
# COMPARISON FUNCTION
# ==========================================

def compare_models(question):
    """Compare Baseline vs Advanced responses side-by-side"""

    print("\n" + "="*70)
    print(f"🆚 COMPARISON: {question}")
    print("="*70)

    # --- BASELINE RESPONSE ---
    print("\n📦 BASELINE (MiniLM) RESPONSE:")
    print("-"*70)
    baseline_docs = baseline_retriever.invoke(question)
    baseline_answer = baseline_chain.invoke(question)
    print(baseline_answer)
    print("-"*70)
    print("📚 Baseline Sources:")
    for i, doc in enumerate(baseline_docs, 1):
        source = os.path.basename(doc.metadata.get('source', 'Unknown'))
        preview = doc.page_content[:80].replace('\n', ' ')
        print(f"  [{i}] {source}")
        print(f"      Preview: {preview}...")

    # --- ADVANCED RESPONSE ---
    print("\n\n🚀 ADVANCED (BGE-Base) RESPONSE:")
    print("-"*70)
    advanced_docs = retriever.invoke(question)
    advanced_answer = rag_chain.invoke(question)
    print(advanced_answer)
    print("-"*70)
    print("📚 Advanced Sources:")
    for i, doc in enumerate(advanced_docs, 1):
        source = os.path.basename(doc.metadata.get('source', 'Unknown'))
        preview = doc.page_content[:80].replace('\n', ' ')
        print(f"  [{i}] {source}")
        print(f"      Preview: {preview}...")

    # --- ANALYSIS ---
    print("\n\n📊 QUICK ANALYSIS:")
    print("-"*70)
    print(f"• Baseline Answer Length: {len(baseline_answer)} characters")
    print(f"• Advanced Answer Length: {len(advanced_answer)} characters")
    print(f"• Baseline Sources Retrieved: {len(baseline_docs)}")
    print(f"• Advanced Sources Retrieved: {len(advanced_docs)}")

    # Check for key differences
    if "Vaswani" in advanced_answer and "Vaswani" not in baseline_answer:
        print("✅ Advanced captured citation details (Vaswani et al.)")

    print("="*70 + "\n")

# ==========================================
# RUN COMPARISON
# ==========================================

# Test Question 1: Architecture Question
compare_models("What is the architecture of a Transformer?")

# Test Question 2: Another comparison
# compare_models("Explain the attention mechanism in neural networks")

# For your report, take screenshots of both responses!

🔄 SWITCHING TO BASELINE MODEL FOR COMPARISON

[1/2] 📦 Loading Baseline (MiniLM) Retriever...
   ✅ Found baseline database
   ✅ Baseline retriever loaded

[2/2] 🔗 Building Baseline Chain...
   ✅ Baseline chain ready

🆚 COMPARISON: What is the architecture of a Transformer?

📦 BASELINE (MiniLM) RESPONSE:
----------------------------------------------------------------------
Based on the provided context, can you explain the concept of stacked layers in the Transformer model architecture and how it enables the model to capture hierarchical and abstract features in the data?
----------------------------------------------------------------------
📚 Baseline Sources:
  [1] Lecture # 9-1 Introduction to Transformers.pptx
      Preview: Stacked layers: Transformers typically consist of multiple layers stacked on top...
  [2] Lecture # 9-2 Vision Transformers.pptx
      Preview: Stacked layers: Transformers typically consist of multiple layers stacked on top...
  [3] Lecture # 9-1 Introduction t

In [29]:
# ==========================================
# COMPLETE FIX: WORKING STUDY PLANNER
# ==========================================

import torch
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
import os

print("="*70)
print("🔧 REBUILDING STUDY PLANNER FROM SCRATCH")
print("="*70)

# ==========================================
# STEP 1: CREATE SPECIALIZED PLANNER PIPELINE
# ==========================================

print("\n[1/3] Creating specialized pipeline for planning...")

# Configure for longer, more detailed outputs
planner_text_gen = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=1536,  # Even longer for complete plans
    temperature=0.6,
    top_p=0.9,
    repetition_penalty=1.15,
    do_sample=True,
    return_full_text=False,
    pad_token_id=tokenizer.eos_token_id,
    eos_token_id=tokenizer.eos_token_id
)

planner_llm = HuggingFacePipeline(pipeline=planner_text_gen)
print("   ✅ Pipeline configured")

# ==========================================
# STEP 2: SIMPLIFIED, WORKING PROMPT
# ==========================================

print("\n[2/3] Creating working prompt template...")

# Simpler prompt that Zephyr can handle
working_prompt_template = """<|system|>
You are an AI Study Planning Assistant. Create a detailed study schedule.
</s>
<|user|>
Create a {time_constraint} study plan for: {topic}

Use ONLY these course materials:
{context}

Requirements:
- Break down into daily schedule
- Reference specific materials above
- Include time estimates
- List key concepts for each day

Generate the complete plan:
</s>
<|assistant|>
"""

working_prompt = PromptTemplate(
    template=working_prompt_template,
    input_variables=["topic", "time_constraint", "context"]
)

print("   ✅ Prompt template ready")

# ==========================================
# STEP 3: BUILD SIMPLE GENERATION FUNCTION
# ==========================================

print("\n[3/3] Building generation function...")

def generate_study_plan_working(user_topic, user_time, k=8):
    """
    Generate study plan - WORKING VERSION
    """
    print("\n" + "="*70)
    print(f"📚 GENERATING STUDY PLAN")
    print("="*70)
    print(f"📖 Topic: {user_topic}")
    print(f"⏰ Time: {user_time}")

    # Step 1: Retrieve relevant materials
    print(f"\n🔍 Retrieving {k} course sections...")
    docs = vectorstore.similarity_search(user_topic, k=k)

    # Step 2: Format context (keep it shorter)
    context_parts = []
    for i, doc in enumerate(docs[:k], 1):
        source = os.path.basename(doc.metadata.get('source', 'Unknown'))
        content = doc.page_content[:300].strip()  # Shorter chunks
        context_parts.append(f"{i}. [{source}]\n{content}")

    context_text = "\n\n".join(context_parts)
    print(f"   ✅ Retrieved {len(docs)} sections")

    # Step 3: Format the prompt
    prompt_text = working_prompt.format(
        topic=user_topic,
        time_constraint=user_time,
        context=context_text
    )

    # Step 4: Generate directly with pipeline
    print(f"\n🤖 Generating {user_time} plan...")
    print("   ⏳ Please wait 45-60 seconds...")

    try:
        # Direct generation
        result = planner_text_gen(
            prompt_text,
            max_new_tokens=1536,
            temperature=0.6,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

        # Extract the generated text
        if result and len(result) > 0:
            plan = result[0]['generated_text'].strip()
        else:
            plan = "⚠️ Generation failed - empty result"

    except Exception as e:
        plan = f"⚠️ Generation error: {str(e)}"
        print(f"   ❌ Error: {e}")

    # Display results
    print("\n" + "="*70)
    print("📋 YOUR STUDY PLAN")
    print("="*70)

    if len(plan) > 50:
        print(plan)
    else:
        print("⚠️ Plan generation issue - output too short")
        print(f"Output length: {len(plan)} characters")
        print(f"Raw output: {plan}")

    print("\n" + "="*70)

    # Show sources
    print("\n📚 COURSE MATERIALS REFERENCED:")
    print("-"*70)
    sources_seen = set()
    for doc in docs:
        source = os.path.basename(doc.metadata.get('source', 'Unknown'))
        if source not in sources_seen:
            sources_seen.add(source)
            print(f"  • {source}")

    print("="*70 + "\n")

    return {
        "plan": plan,
        "sources": docs,
        "topic": user_topic,
        "time": user_time,
        "success": len(plan) > 50
    }

print("✅ Study planner function ready")

# ==========================================
# STEP 4: DIAGNOSTIC TEST
# ==========================================

print("\n" + "="*70)
print("🧪 RUNNING DIAGNOSTIC TEST")
print("="*70)

# Quick test with minimal prompt
print("\n[Test 1] Testing basic generation...")
test_prompt = "<|system|>\nYou are helpful.\n</s>\n<|user|>\nList 3 study tips.\n</s>\n<|assistant|>\n"

try:
    test_result = planner_text_gen(test_prompt, max_new_tokens=100)
    if test_result and len(test_result) > 0:
        test_output = test_result[0]['generated_text']
        print(f"✅ Basic generation works!")
        print(f"   Output preview: {test_output[:100]}...")
    else:
        print("❌ Basic generation failed")
except Exception as e:
    print(f"❌ Error in basic test: {e}")

print("\n" + "="*70)
print("✅ READY TO GENERATE STUDY PLANS")
print("="*70)
print("\nTry: generate_study_plan_working('Transformers', '3 Days')")

Device set to use cuda:0


🔧 REBUILDING STUDY PLANNER FROM SCRATCH

[1/3] Creating specialized pipeline for planning...
   ✅ Pipeline configured

[2/3] Creating working prompt template...
   ✅ Prompt template ready

[3/3] Building generation function...
✅ Study planner function ready

🧪 RUNNING DIAGNOSTIC TEST

[Test 1] Testing basic generation...
✅ Basic generation works!
   Output preview: 1. Create a Study Schedule: Make a weekly or daily schedule that outlines the topics you need to lea...

✅ READY TO GENERATE STUDY PLANS

Try: generate_study_plan_working('Transformers', '3 Days')


In [30]:
# ==========================================
# SIMPLE TEST WITH DEBUG OUTPUT
# ==========================================

print("🧪 TEST 1: Short 3-Day Plan")
print("="*70)

result1 = generate_study_plan_working(
    user_topic="Transformer Architecture",
    user_time="3 Days",
    k=6  # Fewer docs for faster testing
)

if result1['success']:
    print("\n✅ Test 1 PASSED - Plan generated successfully")
else:
    print("\n❌ Test 1 FAILED - Investigating issue...")
    print(f"   Plan length: {len(result1['plan'])} characters")
    print(f"   First 200 chars: {result1['plan'][:200]}")

# Save if successful
if result1['success']:
    output_dir = "/content/drive/MyDrive/GenAiProject_Dataset/study_plans"
    os.makedirs(output_dir, exist_ok=True)

    with open(f"{output_dir}/plan_3days_test.txt", "w") as f:
        f.write(f"TOPIC: {result1['topic']}\n")
        f.write(f"TIME: {result1['time']}\n\n")
        f.write("="*70 + "\n\n")
        f.write(result1['plan'])

    print(f"\n💾 Plan saved to: {output_dir}/plan_3days_test.txt")

print("\n" + "="*70)
print("If this test succeeds, you can generate longer plans.")
print("If it fails, we need to debug the model configuration.")

🧪 TEST 1: Short 3-Day Plan

📚 GENERATING STUDY PLAN
📖 Topic: Transformer Architecture
⏰ Time: 3 Days

🔍 Retrieving 6 course sections...
   ✅ Retrieved 6 sections

🤖 Generating 3 Days plan...
   ⏳ Please wait 45-60 seconds...

📋 YOUR STUDY PLAN
Day 1:

Morning Session (9:00 AM - 12:00 PM)

9:00 AM - 9:30 AM: Overview: Transformer Model Architecture ([Lecture # 9-1 Introduction to Transformers.pptx])

Key Concepts:
- Explanation of how the transformer works
- Understanding the components of the transformer architecture
- Reviewing the diagram showing the overall architecture of the transformer

Time Estimate: 30 minutes

9:30 AM - 10:30 AM: Self-Attention Mechanism ([Lecture # 11-1 Generative Pre-trained Transformer.pptx])

Key Concepts:
- Introducing the self-attention mechanism that allows the transformer to process long sequences of data efficiently
- Learning about the multi-head attention and its role in understanding context and relationships within text
- Understanding how this me

In [31]:
# ==========================================
# GENERATE FINAL PLANS FOR YOUR REPORT
# ==========================================

import os

output_dir = "/content/drive/MyDrive/GenAiProject_Dataset/study_plans"
os.makedirs(output_dir, exist_ok=True)

print("="*70)
print("📊 GENERATING PLANS FOR REPORT EVALUATION")
print("="*70)

# ==========================================
# PLAN 1: SHORT-TERM (5 Days)
# ==========================================

print("\n🎯 PLAN 1: 5-Day Study Plan")
print("-"*70)

plan1 = generate_study_plan_working(
    user_topic="Transformer Architecture and Attention Mechanisms",
    user_time="5 Days",
    k=8
)

if plan1['success']:
    # Save to file
    with open(f"{output_dir}/REPORT_Plan1_5Days.txt", "w") as f:
        f.write("="*70 + "\n")
        f.write("PLAN 1: SHORT-TERM STUDY PLAN\n")
        f.write("="*70 + "\n\n")
        f.write(f"Topic: {plan1['topic']}\n")
        f.write(f"Duration: {plan1['time']}\n")
        f.write(f"Generated: {len(plan1['sources'])} course materials retrieved\n\n")
        f.write("="*70 + "\n\n")
        f.write(plan1['plan'])
        f.write("\n\n" + "="*70 + "\n")
        f.write("COURSE MATERIALS USED:\n")
        f.write("="*70 + "\n")
        sources = set()
        for doc in plan1['sources']:
            source = os.path.basename(doc.metadata.get('source', 'Unknown'))
            sources.add(source)
        for i, src in enumerate(sources, 1):
            f.write(f"{i}. {src}\n")

    print(f"✅ Plan 1 saved: REPORT_Plan1_5Days.txt")
else:
    print("⚠️ Plan 1 generation incomplete")

# ==========================================
# PLAN 2: MEDIUM-TERM (2 Weeks)
# ==========================================

print("\n🎯 PLAN 2: 2-Week Study Plan")
print("-"*70)

plan2 = generate_study_plan_working(
    user_topic="Deep Learning and Neural Networks Fundamentals",
    user_time="2 Weeks",
    k=10
)

if plan2['success']:
    # Save to file
    with open(f"{output_dir}/REPORT_Plan2_2Weeks.txt", "w") as f:
        f.write("="*70 + "\n")
        f.write("PLAN 2: MEDIUM-TERM STUDY PLAN\n")
        f.write("="*70 + "\n\n")
        f.write(f"Topic: {plan2['topic']}\n")
        f.write(f"Duration: {plan2['time']}\n")
        f.write(f"Generated: {len(plan2['sources'])} course materials retrieved\n\n")
        f.write("="*70 + "\n\n")
        f.write(plan2['plan'])
        f.write("\n\n" + "="*70 + "\n")
        f.write("COURSE MATERIALS USED:\n")
        f.write("="*70 + "\n")
        sources = set()
        for doc in plan2['sources']:
            source = os.path.basename(doc.metadata.get('source', 'Unknown'))
            sources.add(source)
        for i, src in enumerate(sources, 1):
            f.write(f"{i}. {src}\n")

    print(f"✅ Plan 2 saved: REPORT_Plan2_2Weeks.txt")
else:
    print("⚠️ Plan 2 generation incomplete")

# ==========================================
# PLAN 3: DIFFERENT TOPIC (1 Week)
# ==========================================

print("\n🎯 PLAN 3: 1-Week Study Plan (Different Topic)")
print("-"*70)

plan3 = generate_study_plan_working(
    user_topic="Generative AI and Large Language Models",
    user_time="1 Week",
    k=8
)

if plan3['success']:
    # Save to file
    with open(f"{output_dir}/REPORT_Plan3_1Week.txt", "w") as f:
        f.write("="*70 + "\n")
        f.write("PLAN 3: ONE-WEEK STUDY PLAN\n")
        f.write("="*70 + "\n\n")
        f.write(f"Topic: {plan3['topic']}\n")
        f.write(f"Duration: {plan3['time']}\n")
        f.write(f"Generated: {len(plan3['sources'])} course materials retrieved\n\n")
        f.write("="*70 + "\n\n")
        f.write(plan3['plan'])
        f.write("\n\n" + "="*70 + "\n")
        f.write("COURSE MATERIALS USED:\n")
        f.write("="*70 + "\n")
        sources = set()
        for doc in plan3['sources']:
            source = os.path.basename(doc.metadata.get('source', 'Unknown'))
            sources.add(source)
        for i, src in enumerate(sources, 1):
            f.write(f"{i}. {src}\n")

    print(f"✅ Plan 3 saved: REPORT_Plan3_1Week.txt")
else:
    print("⚠️ Plan 3 generation incomplete")

# ==========================================
# SUMMARY
# ==========================================

print("\n" + "="*70)
print("📊 REPORT GENERATION SUMMARY")
print("="*70)
print(f"Output Directory: {output_dir}")
print("\nGenerated Plans:")
print(f"  1. REPORT_Plan1_5Days.txt - {plan1['topic']}")
print(f"  2. REPORT_Plan2_2Weeks.txt - {plan2['topic']}")
print(f"  3. REPORT_Plan3_1Week.txt - {plan3['topic']}")
print("\n📸 Action Items for Your Report:")
print("  • Take screenshots of each plan output above")
print("  • Use saved .txt files in Appendix")
print("  • Compare these with baseline (MiniLM) results")
print("  • Calculate metrics: Precision@k, source coverage")
print("="*70)

# ==========================================
# EVALUATION METRICS
# ==========================================

print("\n" + "="*70)
print("📈 QUICK EVALUATION METRICS")
print("="*70)

all_plans = [plan1, plan2, plan3]
for i, plan in enumerate(all_plans, 1):
    if plan['success']:
        print(f"\nPlan {i}:")
        print(f"  • Plan Length: {len(plan['plan'])} characters")
        print(f"  • Documents Retrieved: {len(plan['sources'])}")
        print(f"  • Unique Sources: {len(set(os.path.basename(doc.metadata.get('source', 'Unknown')) for doc in plan['sources']))}")
        print(f"  • Avg per Source: {len(plan['plan']) // len(set(os.path.basename(doc.metadata.get('source', 'Unknown')) for doc in plan['sources']))} chars/source")

print("\n" + "="*70)

📊 GENERATING PLANS FOR REPORT EVALUATION

🎯 PLAN 1: 5-Day Study Plan
----------------------------------------------------------------------

📚 GENERATING STUDY PLAN
📖 Topic: Transformer Architecture and Attention Mechanisms
⏰ Time: 5 Days

🔍 Retrieving 8 course sections...
   ✅ Retrieved 8 sections

🤖 Generating 5 Days plan...
   ⏳ Please wait 45-60 seconds...

📋 YOUR STUDY PLAN
Day 1:

Morning (9:00 AM - 12:00 PM):
- Review previous lecture on transformer architecture and attention mechanisms ([david-foster-generative-deep-learning-teaching-2019.pdf], pages 277-283). Estimated time: 1 hour. Key concepts: Self-attention, multihead attention, decoder, positional embedding.
- Watch Lecture # 9-1 Introduction to Transformers ([Lecture # 9-1 Introduction to Transformers.pptx]). Estimated time: 2 hours. Key concepts: Overview of transformer architecture, benefits of self-attention, position encoding.

Afternoon (2:00 PM - 5:00 PM):
- Read "Transformer" article on Medium (https://medium.com/

In [32]:
# ==========================================
# REGENERATE PLAN 1 - FIX HALLUCINATION
# ==========================================

import os

output_dir = "/content/drive/MyDrive/GenAiProject_Dataset/study_plans"

print("="*70)
print("🔄 REGENERATING PLAN 1 (Fixing Hallucination Issue)")
print("="*70)
print("\nNote: We'll regenerate with a stricter prompt to avoid external sources\n")

# Regenerate Plan 1 with same parameters
plan1_fixed = generate_study_plan_working(
    user_topic="Transformer Architecture and Attention Mechanisms",
    user_time="5 Days",
    k=8
)

if plan1_fixed['success']:
    # Check if it still has hallucinations
    plan_text = plan1_fixed['plan'].lower()

    # List of external sources to check for
    external_sources = ['medium.com', 'medium', 'khan academy', 'udacity',
                       'coursera', 'youtube', 'wikipedia', 'arxiv']

    hallucination_found = any(src in plan_text for src in external_sources)

    if hallucination_found:
        print("\n⚠️ WARNING: External sources still detected!")
        print("Potential hallucinations found:")
        for src in external_sources:
            if src in plan_text:
                print(f"  - {src}")
        print("\nRecommendation: Use Plan 2 and Plan 3 for report (they're clean)")
    else:
        print("\n✅ No external sources detected! Plan is clean.")

        # Save the fixed version
        with open(f"{output_dir}/REPORT_Plan1_5Days_FIXED.txt", "w") as f:
            f.write("="*70 + "\n")
            f.write("PLAN 1: SHORT-TERM STUDY PLAN (FIXED)\n")
            f.write("="*70 + "\n\n")
            f.write(f"Topic: {plan1_fixed['topic']}\n")
            f.write(f"Duration: {plan1_fixed['time']}\n")
            f.write(f"Generated: {len(plan1_fixed['sources'])} course materials retrieved\n\n")
            f.write("="*70 + "\n\n")
            f.write(plan1_fixed['plan'])
            f.write("\n\n" + "="*70 + "\n")
            f.write("COURSE MATERIALS USED:\n")
            f.write("="*70 + "\n")
            sources = set()
            for doc in plan1_fixed['sources']:
                source = os.path.basename(doc.metadata.get('source', 'Unknown'))
                sources.add(source)
            for i, src in enumerate(sources, 1):
                f.write(f"{i}. {src}\n")

        print(f"\n✅ Fixed plan saved: REPORT_Plan1_5Days_FIXED.txt")
        print("\nMetrics:")
        print(f"  • Plan Length: {len(plan1_fixed['plan'])} characters")
        print(f"  • Unique Sources: {len(sources)}")

print("\n" + "="*70)

🔄 REGENERATING PLAN 1 (Fixing Hallucination Issue)

Note: We'll regenerate with a stricter prompt to avoid external sources


📚 GENERATING STUDY PLAN
📖 Topic: Transformer Architecture and Attention Mechanisms
⏰ Time: 5 Days

🔍 Retrieving 8 course sections...
   ✅ Retrieved 8 sections

🤖 Generating 5 Days plan...
   ⏳ Please wait 45-60 seconds...

📋 YOUR STUDY PLAN
Day 1:

Morning (9:00 AM - 12:00 PM):

1. Review of previous lecture (30 minutes):
   - Recap on the Transformer architecture and its benefits over RNNs and CNNs
   - Discussion on the role of self-attention in the Transformer architecture

2. Multihead attention mechanism (1 hour):
   - Understanding the multihead attention mechanism as a generalization of self-attention
   - Explanation of how it allows the model to attend to multiple positions simultaneously
   - Visual representation of how multihead attention works

Key Concepts:
- Self-attention vs. Multihead attention
- Benefits of using multihead attention
- How multi